In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score

In [ ]:
# 1. Load MNIST (Handwritten Digits 0-9)
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Using a subset for rapid prototyping
X_train = train_set.data[:5000].float() / 255.0
y_train = train_set.targets[:5000].numpy()
X_test = test_set.data[:1000].float() / 255.0
y_test = test_set.targets[:1000].numpy()

In [ ]:
# 2. Hyperparameters
input_size = 28   # 28 pixels per row
res_size = 1000   # Larger reservoir for complex handwriting
spectral_radius = 0.9

# 3. Fixed Reservoir Weights
Win = (np.random.rand(res_size, input_size + 1) - 0.5) * 0.2
Wres = np.random.randn(res_size, res_size)
Wres *= spectral_radius / np.max(np.abs(np.linalg.eigvals(Wres)))

In [ ]:
def get_reservoir_states(data):
    all_states = []
    for img in data:
        x = np.zeros((res_size, 1))
        # Feed the image row-by-row (28 time steps)
        for row in img:
            u = row.numpy().reshape(-1, 1)
            # Standard ESN update: x = tanh(Win*u + Wres*x)
            x = np.tanh(np.dot(Win, np.vstack(([1.0], u))) + np.dot(Wres, x))
        all_states.append(x.flatten())
    return np.array(all_states)

In [ ]:
# 4. Training the Readout
print("Processing training images...")
train_states = get_reservoir_states(X_train)
readout = Ridge(alpha=1.0)
readout.fit(train_states, y_train) # Training just the final mapping

In [ ]:
# 5. Testing
print("Testing on unseen handwriting...")
test_states = get_reservoir_states(X_test)
preds = np.round(readout.predict(test_states)).astype(int)
preds = np.clip(preds, 0, 9)

print(f"Accuracy: {accuracy_score(y_test, preds) * 100:.2f}%")